<a href="https://colab.research.google.com/github/robertbarcik/vector_databases-tutorial/blob/main/3.%20LlamaIndex/1_LlamaIndex_RAG_and_Document_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LlamaIndex: RAG in Five Lines, and the Problem It Does Not Solve

In the ChromaDB notebooks you built every step of RAG by hand: load the text, cut it into chunks, embed, store, retrieve, build a prompt, call the model. That is the right way to learn it. It is also a lot of code to write again for every project.

**LlamaIndex** is a framework that packs those steps into a few objects. In this notebook you first get a working question-answering system over a 23-page PDF in five lines, then open it up and find every step you already know. After that comes the part no framework does for you by default: a question about a **table** gets a confident, wrong answer, and you fix it by parsing the document properly.

## Setup

Two keys. The **OpenAI** key is needed throughout (secret `OPENAI_API_KEY`). The **LlamaCloud** key is used in the second half to parse the PDF with LlamaParse; it is free at [cloud.llamaindex.ai](https://cloud.llamaindex.ai) (secret `LLAMA_CLOUD_API_KEY`). If you do not have one, press Enter when asked: the notebook then uses a copy of the parsed document stored in the course repository, and everything still runs.

In [1]:
%pip install -q llama-index-core==0.14.26 llama-index-llms-openai==0.9.0 llama-index-embeddings-openai==0.8.0 llama-index-readers-file==0.8.0 llama-index-vector-stores-chroma==0.7.0 llama-cloud==2.19.0 chromadb==1.5.9 openai==3.51.0   # Colab installs here; locally, requirements.txt already covers it

import os, logging, pathlib, urllib.request

for name in ("httpx", "httpx2", "openai", "llama_cloud"):   # otherwise every HTTP request is logged into the cell output
    logging.getLogger(name).setLevel(logging.WARNING)

def colab_secret(name):
    """The key from Colab secrets, or None when we are not in Colab or there is no such secret."""
    try:
        from google.colab import userdata
        for attempt in range(12):                   # Colab shows a "Grant access" dialog the first time and waits only
            try:                                    # 10 seconds for your click. We keep asking for about two minutes.
                return userdata.get(name)
            except userdata.TimeoutException:
                pass
    except Exception:
        pass

def get_key(name, label):
    """Colab secret -> environment variable -> a prompt. Returns the key without stray spaces or newlines."""
    key = colab_secret(name) or os.environ.get(name)
    while not isinstance(key, str) or not key.strip():      # nothing found: type it in
        from getpass import getpass
        typed = getpass(f"{label}: ")
        key = typed if isinstance(typed, str) else colab_secret(name)   # a late click on "Grant access" lands here: read the secret again
    return key.strip()

os.environ["OPENAI_API_KEY"] = get_key("OPENAI_API_KEY", "OpenAI API key")

# The LlamaCloud key is optional: Colab secret -> environment variable -> a prompt where Enter means "skip".
llama_key = colab_secret("LLAMA_CLOUD_API_KEY") or os.environ.get("LLAMA_CLOUD_API_KEY")
if not isinstance(llama_key, str) or not llama_key.strip():
    from getpass import getpass
    typed = getpass("LlamaCloud API key (press Enter to use the parsed copy from the course repository): ")
    llama_key = typed if isinstance(typed, str) else colab_secret("LLAMA_CLOUD_API_KEY")
llama_key = (llama_key or "").strip()

# Data files: next to this notebook if you downloaded the course folder; fetched from GitHub otherwise (Colab).
RAW = "https://raw.githubusercontent.com/robertbarcik/vector_databases-tutorial/main/3.%20LlamaIndex/"
for file_name in ("livestock_poultry.pdf", "livestock_poultry_parsed.md"):
    if not pathlib.Path(file_name).exists():
        urllib.request.urlretrieve(RAW + file_name, file_name)

from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-6-luna")                               # the model that writes the answers
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")  # the model that makes the vectors

try:                                                # one tiny request now, so a wrong key fails here with a clear message
    Settings.llm.complete("Say OK.")
except Exception as error:
    raise RuntimeError(f"The test request to OpenAI failed: {type(error).__name__}: {error}\n"
                       "Check the key (in Colab: the key icon on the left, secret OPENAI_API_KEY) and run this cell again.") from None

print("Ready. LlamaCloud key:", "yes" if llama_key else "no, the parsing step will use the parsed copy from the repository")

Note: you may need to restart the kernel to use updated packages.


Ready. LlamaCloud key: yes


## 3.1 RAG in five lines, and what is inside them

The document is a report of the US Department of Agriculture on world meat markets: 23 pages, half of them prose, half of them tables of numbers. A typical business document.

Read the next cell before you run it. Each line is one step of the pipeline you built by hand.

In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(input_files=["livestock_poultry.pdf"]).load_data()   # 1. load the file
index = VectorStoreIndex.from_documents(documents)                                     # 2. chunk, embed, store
query_engine = index.as_query_engine()                                                 # 3. retriever + prompt + model
response = query_engine.query("Why is China's beef import growth expected to slow?")   # 4. ask
print(response)                                                                        # 5. read

China’s beef import growth is expected to slow because economic uncertainty and cautious consumer spending are likely to constrain demand. An ongoing safeguard investigation—which could lead to extra duties or import limits—is also adding uncertainty for suppliers.


### 🔍 What just happened?

A grounded answer from a PDF, in five lines. Nothing in them is new to you, only shorter:

- `SimpleDirectoryReader(...).load_data()` opened the PDF and pulled out the text, one **document** per page.
- `VectorStoreIndex.from_documents(...)` cut the pages into chunks, sent them to the embedding model and kept the vectors in memory. This is your chunking function plus `collection.add()`.
- `as_query_engine()` wired together a retriever, a prompt template and the chat model. This is your `retrieve()` plus `build_prompt()` plus the call to OpenAI.

The models did not come from nowhere: the setup cell put them into `Settings`, LlamaIndex's place for defaults. Always set them yourself. A framework's built-in defaults age quickly, and you want to know which model answers your users.

### What the five lines did

Short code is pleasant until an answer is wrong and you have to find out why. So let's open the box. First the loading step: how many documents, and what does one look like?

In [3]:
print(len(documents), "documents, one per page\n")
page = documents[2]
print("metadata:", {k: page.metadata[k] for k in ("page_label", "file_name")})
print("\ntext:", page.text[:400].strip(), "...")

23 documents, one per page

metadata: {'page_label': '3', 'file_name': 'livestock_poultry.pdf'}

text: Livestock and Poultry: World Markets and Trade 
 
China Beef Import Growth to Slow Amid Global Uncertainty 
2025 China beef imports are forecast to increase 2 percent, a slower growth rate than in previous 
years. Economic uncertainty and cautious consumer spending will constrain beef demand. 
Additionally, the largest beef suppliers are keenly awaiting the results of a special safeguard 
in ...


Next the chunks. LlamaIndex calls them **nodes**, and the tool that cuts them is the `SentenceSplitter`. It has the two knobs you tuned by hand in the chunking notebook, `chunk_size` and `chunk_overlap`, counted in tokens, and it prefers to cut between sentences. We pass our own splitter so that the settings are visible instead of hidden defaults.

In [4]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)
print(len(documents), "pages became", len(nodes), "nodes\n")
print("one node, with the metadata it inherited from its page:")
print("  ", {k: nodes[5].metadata[k] for k in ("page_label", "file_name")})
print("  ", nodes[5].text[:200].strip().replace("\n", " "), "...")

index = VectorStoreIndex(nodes)        # the same index as before, now built from chunks we control

23 pages became 33 nodes

one node, with the metadata it inherited from its page:
   {'page_label': '6', 'file_name': 'livestock_poultry.pdf'}
   Livestock and Poultry: World Markets and Trade    Global production in 2025 is forecast virtually unchanged from 2024 at 61.6 million tons as falling  production in the United States and the European ...


And the retrieval. Every answer carries its **source nodes**: the chunks that were retrieved and put into the prompt, with their similarity scores. `similarity_top_k` is the `n_results` you know from Chroma.

In [5]:
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query("Why is China's beef import growth expected to slow?")
print(response, "\n")

for source in response.source_nodes:
    print(f"   score {source.score:.3f}  page {source.metadata['page_label']:>2}  {source.text[:90].strip().replace(chr(10), ' ')} ...")

China’s beef import growth is forecast to slow to 2 percent in 2025 as economic uncertainty and cautious consumer spending weigh on demand. An ongoing safeguard investigation could also bring additional duties or import limits, adding uncertainty to trade. 

   score 0.702  page  3  Livestock and Poultry: World Markets and Trade    China Beef Import Growth to Slow Amid Gl ...
   score 0.640  page  5  Livestock and Poultry: World Markets and Trade  Another factor that may impact total China ...
   score 0.630  page  4  Livestock and Poultry: World Markets and Trade  result of the China-Australia Free Trade A ...


### 🔍 What just happened?

The box has no secrets. Pages became nodes with a splitter you configured, the question was embedded and compared with the node vectors, the three closest nodes went into a prompt, and the model answered from them. The scores are similarities, so bigger is better, as in Pinecone.

`response.source_nodes` is the habit to keep from this section. Whenever an answer looks odd, print the sources first. In most cases the model was fine and the retrieved text was the problem. The next section is exactly such a case.

### 🎯 Mini-task

Ask "What is forecast for global pork production in 2025?" and print the source nodes. Then rebuild the nodes with `chunk_size=128` and ask again. How do the sources and the answer change?

In [6]:
# YOUR CODE HERE

## 3.2 Parsing sets the ceiling

### A question about a table

Half of this report is tables. Page 9 holds the cattle stocks of the biggest countries, year by year. In the PDF, the row for Brazil reads: 193,195 for 2021, 193,780 for 2022, **194,365 for 2023**, then 192,572, 186,875 and 186,875. Let's ask for one of these numbers.

In [7]:
table_question = "How many cattle did Brazil have at the beginning of 2023, according to the cattle stocks table? Answer with the number from the table."

response = query_engine.query(table_question)
print(response, "\n")
print("pages used:", [source.metadata["page_label"] for source in response.source_nodes])

192,572 

pages used: ['9', '11', '10']


The right page was retrieved, so retrieval is not to blame. Compare the number with the one from the PDF above. Then look at what the model was actually given: the text that the reader pulled out of page 9.

In [8]:
print(documents[8].text[:620])

Cattle Stocks - Top Countries Summary
(in 1,000 head)
2021 2022 2023 2024 2025
Oct
2025
Apr
          none
Total Cattle Beg. Stks
307,490307,490307,420307,400306,700305,500    India
186,875186,875192,572194,365193,780193,195    Brazil
104,900104,000105,090102,16098,17295,621    China
71,82272,30073,74574,80875,70576,551    European Union
52,37053,20052,80054,10053,40053,540    Argentina
27,26027,02027,08025,80023,94423,021    Australia
17,96517,73517,84017,76317,31417,000    Mexico
17,05016,65017,28517,43517,79817,953    Russia
11,80511,80511,36611,79511,64611,966    Uruguay
10,94010,93511,01511,24511,51511,515  


### 🔍 What just happened?

This is the most common way RAG fails on business documents, and it has nothing to do with the model.

A PDF does not store a table. It stores characters at positions on a page. The plain reader collected them in the order it found them, and the result is what you see: the six numbers of a row glued together without spaces, in **reversed** order, with the country name at the end. The year header sits a few lines above with no connection to the columns.

The model did its best with that. It split the digits into numbers, matched them to the years in the order they appear, and reported a value from the wrong column, in a confident sentence. No warning, no hesitation. A reader of the answer has no way to notice.

Garbage in, confident answer out. Better prompts or a stronger model will not repair this. The text has to be extracted properly in the first place.

### Parsing the document properly

**Document parsing** has become a product category of its own. A parser looks at the page the way you do, as a layout, and writes out what it sees in a structured format: headings as headings, tables as tables. **LlamaParse**, from the makers of LlamaIndex, is one such service; open-source and other hosted parsers exist and work on the same principle.

We send the PDF and ask for Markdown. `tier` chooses how much effort the service spends per page: `fast` is plain text, `cost_effective` keeps headings and tables, `agentic` is for hard layouts. The free plan covers a few thousand pages a month at the middle tier.

In [9]:
from llama_cloud import LlamaCloud

PAGE_BREAK = "\n\n<!-- page break -->\n\n"
parsed_pages = None

if llama_key:
    for base_url in ("https://api.cloud.eu.llamaindex.ai", "https://api.cloud.llamaindex.ai"):   # a key belongs to one region: EU or US
        try:
            client = LlamaCloud(api_key=llama_key, base_url=base_url)
            result = client.parsing.parse(
                upload_file=pathlib.Path("livestock_poultry.pdf"),
                tier="cost_effective",
                version="latest",
                expand=["markdown"],
            )
            parsed_pages = [page.markdown for page in result.markdown.pages]
            print("parsed by LlamaParse:", len(parsed_pages), "pages")
            break
        except Exception as error:
            print(f"LlamaParse at {base_url} did not work ({type(error).__name__}).")

if parsed_pages is None:                             # no key, wrong key, no credits left, or the service is busy
    parsed_pages = pathlib.Path("livestock_poultry_parsed.md").read_text(encoding="utf-8").split(PAGE_BREAK)
    print("using the parsed copy from the course repository:", len(parsed_pages), "pages")

print("\npage 9 as Markdown:\n")
print(parsed_pages[8][:900], "...")

parsed by LlamaParse: 23 pages

page 9 as Markdown:

# **Cattle Stocks - Top Countries Summary**
### **(in 1,000 head)**

<table><tr><td></td><th>2021</th><th>2022</th><th>2023</th><th>2024</th><th>2025 Oct</th><th>2025 Apr</th></tr><tr><td>Total Cattle Beg. Stks</td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td>India</td><td>305,500</td><td>306,700</td><td>307,400</td><td>307,420</td><td>307,490</td><td>307,490</td></tr><tr><td>Brazil</td><td>193,195</td><td>193,780</td><td>194,365</td><td>192,572</td><td>186,875</td><td>186,875</td></tr><tr><td>China</td><td>95,621</td><td>98,172</td><td>102,160</td><td>105,090</td><td>104,000</td><td>104,900</td></tr><tr><td>European Union</td><td>76,551</td><td>75,705</td><td>74,808</td><td>73,745</td><td>72,300</td><td>71,822</td></tr><tr><td>Argentina</td><td>53,540</td><td>53,400</td><td>54,100</td><td>52,800</td><td>53,200</td><td>52,370</td></tr><tr><td>Australia</td><td>23,02 ...


Same page, now with a real table: a header row with the years, one row per country, every number in its own cell. (The parser writes tables as HTML tags inside the Markdown, because HTML can express merged cells and nested headers. Language models read it without trouble.)

Now the same pipeline as before, built from the parsed pages. A parsed page is just text, so we wrap each one in a `Document` with its page number and reuse the same splitter.

In [10]:
from llama_index.core import Document

parsed_documents = [
    Document(text=text, metadata={"file_name": "livestock_poultry.pdf", "page_label": str(number)})
    for number, text in enumerate(parsed_pages, start=1)
]
parsed_index = VectorStoreIndex(splitter.get_nodes_from_documents(parsed_documents))
parsed_engine = parsed_index.as_query_engine(similarity_top_k=3)

print("plain reader :", query_engine.query(table_question))
print("parsed pages :", parsed_engine.query(table_question))

plain reader : 192,572


parsed pages : 194,365


### 🔍 What just happened?

Same question, same models, same splitter, same retrieval settings. The only thing that changed is how the text was taken out of the PDF, and the answer went from wrong to right: 194,365 thousand head, the number in the table.

That is the lesson of this notebook. In a RAG system over real documents, **the quality of parsing sets the ceiling** for everything that follows. Before you tune chunk sizes or change models, open a few retrieved chunks and read them. If they look like page 9 did before parsing, fix the parsing first.

One honest limit: a long table can still be cut in two by the splitter, and the second half then has no header row. Parsing fixes the text; chunking still needs care.

### 🎯 Mini-task

Ask both engines: "What were Australia's total cattle beginning stocks in 2021 and in the April 2025 forecast?" The PDF says 23,021 and 27,260. Which engine swaps them, and why does the reversed row order explain it?

In [11]:
# YOUR CODE HERE

## 🛠️ Developer corner: keeping the index in Chroma

So far the vectors lived in memory and disappear with the notebook. LlamaIndex does not insist on its own storage. It can sit on top of the vector databases you already know, and the index then becomes a thin layer over a Chroma collection (or a Pinecone index).

The pattern is always the same: open the store the way you would without LlamaIndex, wrap it in a `VectorStore` object, and hand that to the index.

In [12]:
import chromadb
from llama_index.core import StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma = chromadb.PersistentClient(path="./chroma_db")
collection = chroma.get_or_create_collection("livestock_report")
vector_store = ChromaVectorStore(chroma_collection=collection)

if collection.count() == 0:                          # first run: embed the parsed pages and write them into Chroma
    VectorStoreIndex(
        splitter.get_nodes_from_documents(parsed_documents),
        storage_context=StorageContext.from_defaults(vector_store=vector_store),
    )
print("rows in the Chroma collection:", collection.count())

stored_index = VectorStoreIndex.from_vector_store(vector_store)     # any later session: open the index, nothing is embedded again
print(stored_index.as_query_engine(similarity_top_k=3).query(table_question))

rows in the Chroma collection: 67


194,365


### 🔍 What just happened?

The nodes went into an ordinary Chroma collection on disk, and `from_vector_store()` opened an index over it without embedding anything again. Tomorrow, or in another program, those two lines are all you need.

It is the same collection type you worked with in the first module. You can open it with plain `chromadb`, look at it with `collection.peek()`, filter it with `where`. The framework is a convenience layer over the store. It does not lock your data in.

**🛠️ Stretch.** Open the collection without LlamaIndex: `collection.get(limit=2, include=["documents", "metadatas"])`. Which metadata did LlamaIndex store with each chunk?

In [13]:
# YOUR CODE HERE

## 🛠️ Developer corner: let the model label the chunks

One more idea that frameworks make easy. A chunk on its own often lacks context: "Total Foreign 843,902" means little without knowing which table it came from. An **extractor** sends each chunk to the language model and stores the answer as metadata next to the chunk: a title, a summary, or the questions the chunk can answer. That metadata is embedded together with the text, so the chunk becomes easier to find.

It costs one model call per chunk, so we try it on two pages only.

In [14]:
from llama_index.core.extractors import QuestionsAnsweredExtractor
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(transformations=[splitter, QuestionsAnsweredExtractor(questions=2, show_progress=False)])
labelled = pipeline.run(documents=parsed_documents[8:10], show_progress=False)   # pages 9 and 10 only
print(len(labelled), "chunks labelled\n")

for node in labelled[:2]:
    print("page", node.metadata["page_label"], "| chunk starts:", node.text[:70].strip().replace("\n", " "), "...")
    print(node.metadata["questions_this_excerpt_can_answer"].strip(), "\n")

7 chunks labelled

page 9 | chunk starts: # **Cattle Stocks - Top Countries Summary** ### **(in 1,000 head)**  < ...
1. What were India’s cattle stocks in 2025 Oct, in 1,000 head?
2. How did Brazil’s cattle stocks change from 2024 to 2025 Oct, in 1,000 head? 

page 9 | chunk starts: 020</td><td>27,260</td></tr><tr><td>Mexico</td><td>17,000</td><td>17,3 ...
1. What figure is reported for Canada in the rightmost numeric column?
2. What is the “Total Foreign” figure in the fourth numeric column? 



### 🔍 What just happened?

For each chunk the model wrote down questions that the chunk can answer, and LlamaIndex stored them in the chunk's metadata. When a user later asks something similar, the question now resembles the chunk much more closely than a bare table of numbers would.

Look at the second chunk, though. It starts in the middle of the table, so it has no header row. Read its questions: without the years, the model can only refer to the columns by their position. That is the limit mentioned in section 3.2 made visible: the splitter cut a table in two. Labels cannot repair a chunk that lost its header.

The idea has a general name, **enriching chunks with context at ingestion time**, and it is one of the cheaper ways to improve retrieval on documents full of tables and fragments. The price is one model call per chunk when you load the data, nothing at question time.

## What you take with you

- LlamaIndex packs the RAG pipeline into three steps: **load** documents, build an **index** (chunk, embed, store), ask a **query engine** (retrieve, prompt, generate). Each one is a step you have already written by hand.
- Set the models in `Settings` yourself, pass your own `SentenceSplitter`, and read `response.source_nodes` whenever an answer looks odd.
- **Parsing sets the ceiling.** A plain PDF reader scrambles tables, and the model then answers confidently from scrambled text. A layout-aware parser such as LlamaParse turns the page into structured Markdown first.
- The index can live in the vector database you already use; with Chroma underneath, nothing is locked into the framework.
- Extractors let the model add context to each chunk when the data is loaded.

When is a framework worth it? When you need many file formats, ready-made connectors and standard pipelines quickly. When you need full control over one specific pipeline, the hand-written version from the first module is short, readable, and has no surprises.